In [1]:
# para manipulación y análisis de datos estructurados
import pandas as pd

# es el módulo principal de matplotlib, 
# una librería de visualización de gráficos estáticos en 2D.
import matplotlib.pyplot as plt

# Es una magic command de Jupyter Notebook
%matplotlib inline

# es una librería de visualización construida sobre matplotlib, 
# con una sintaxis más simple y gráficos más bonitos
import seaborn as sns

In [2]:
#!pip install pyarrow

In [ ]:
# Identifique el método utilizado para capturar los datos, por ejemplo, ODBC.
transactions = pd.read_parquet('../transactions_agg.parquet') #Data sin duplicados

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">se multiplica por 50000 dado que los datos son anonimizados

In [4]:
transactions.price = transactions.price*50000

In [5]:
pd.set_option('display.float_format', '{:.6f}'.format) 
transactions.describe()

,article_id,sales_channel_id,price
count,28583889.000000,28583889.000000,28583889.000000
mean,697398489.243995,1.682593,1547.455586
std,131727610.245217,0.465467,1278.020702
min,108775015.000000,1.000000,0.847458
25%,633377003.000000,1.000000,846.610169
50%,714790003.000000,2.000000,1270.338983
75%,787216001.000000,2.000000,1755.084746
max,956217002.000000,2.000000,283322.033898


<a id='knn'></a>
<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px">
Churn analysis

In [6]:
import dask.dataframe as dd

In [7]:
# filtrar columnas
dd_knn = dd.from_pandas(transactions[["customer_id", "price", "t_dat"]], npartitions=2)

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">
Frecuencia


In [8]:
# Agrupar por cliente y contar transacciones
frecuency = (
    dd_knn
    .groupby("customer_id")
    .agg(Frequency=("customer_id", "count"))
    .reset_index()
)

# Si quieres materializar el resultado en pandas
frecuency_result = frecuency.compute()

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">
Cantidad

In [9]:
# Agrupar por cliente y sumar la columna price
amount = (
    dd_knn
    .groupby("customer_id")
    .agg(Sold=("price", "sum"))
    .reset_index()
)

# Si quieres traerlo a pandas
amount_result = amount.compute()

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">
RECENCIA

In [10]:
# Asegurar fecha
dd_knn["t_dat"] = dd.to_datetime(dd_knn["t_dat"])



In [11]:
#fecha de referencia
fecha_referencia = dd_knn["t_dat"].max().compute()

In [12]:
fecha_referencia

Timestamp('2020-09-22 00:00:00')

In [13]:
# Última compra por cliente
last_purchase = dd_knn.groupby("customer_id")["t_dat"].max().reset_index()
last_purchase["Recency"] = (fecha_referencia - last_purchase["t_dat"]).dt.days

In [14]:
last_purchase.compute()

,customer_id,t_dat,Recency
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2020-09-05,17
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,2020-07-08,76
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2020-09-15,7
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,2019-06-09,471
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,2020-08-12,41
...,...,...,...
1362276,ffff11185bb6dc52d546424a28657dcb3c891b1f57e27b...,2020-06-25,89
1362277,ffff2d1849db66617499febae392fb5e9335ebf160de0e...,2019-11-22,305
1362278,ffff7d65748db4d52e48b74c8f83ccb0029fc3bbafa511...,2019-10-20,338
1362279,ffffd7744cebcf3aca44ae7049d2a94b87074c3d4ffe38...,2020-06-22,92


<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">
Consolidado

In [15]:
# 1. Unir Frecuencia y Cantidad (Monetario/Sold)
retail = frecuency.merge(amount, on="customer_id", how="inner")

# Unir con recency
retail = retail.merge(last_purchase[["customer_id", "Recency"]], on="customer_id", how="inner")

# Si quieres ver el resultado
retail_result = retail.compute()

In [16]:
retail_result.head()

,customer_id,Frequency,Sold,Recency
0,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,15,35238.983051,7
1,00007d2de826758b65a93dd24ce629ed66842531df6699...,113,191180.508475,132
2,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...,26,31396.610169,174
3,0008968c0d451dbc5a9968da03196fe20051965edde741...,61,75656.779661,8
4,000aa7f0dc06cd7174389e76c9e132a67860c5f65f9706...,37,74527.966102,160


In [17]:
df = retail_result.copy()

In [19]:
# Percentiles
df['recency_percentile'] = df['Recency'].rank(method='min') / len(df)
df['frequency_percentile'] = df['Frequency'].rank(method='min') / len(df)
df['monetary_percentile'] = df['Sold'].rank(method='min') / len(df) 

In [20]:
# Asignación de scores
def assign_recency_score(p):
    if p <= 0.2: return 5
    elif p <= 0.4: return 4
    elif p <= 0.6: return 3
    elif p <= 0.8: return 2
    else: return 1

def assign_frequency_score(p):
    if p <= 0.2: return 1
    elif p <= 0.4: return 2
    elif p <= 0.6: return 3
    elif p <= 0.8: return 4
    else: return 5

def assign_monetary_score(p):
    if p <= 0.2: return 1
    elif p <= 0.4: return 2
    elif p <= 0.6: return 3
    elif p <= 0.8: return 4
    else: return 5

In [22]:
# Aplicar funciones
df['R_score'] = df['recency_percentile'].apply(assign_recency_score)
df['F_score'] = df['frequency_percentile'].apply(assign_frequency_score)
df['M_score'] = df['monetary_percentile'].apply(assign_monetary_score)

In [ ]:
# Definir segmento churn
def map_churn(row):
    # Regla : baja R + bajo F + bajo M = churn
    if row['R_score'] <= 2 and row['F_score'] <= 2 and row['M_score'] <= 2:
        return 'churn'
    else:
        return 'no_churn'

df['churn_segment'] = df.apply(map_churn, axis=1)

In [25]:
# Cantidad de clientes por segmento
print(df['churn_segment'].value_counts())

churn_segment
no_churn    1039740
churn        322541
Name: count, dtype: int64


In [26]:
# % de churn
print(df['churn_segment'].value_counts(normalize=True) * 100)

churn_segment
no_churn   76.323460
churn      23.676540
Name: proportion, dtype: float64


In [27]:
# Top 10 clientes churn por gasto
df[df['churn_segment']=='churn'].sort_values('Sold', ascending=False).head(10)

,customer_id,Frequency,Sold,Recency,recency_percentile,frequency_percentile,monetary_percentile,R_score,F_score,M_score,churn_segment
953734,66ae48eda84643c8dcd8f629272f30267118ac4483ae57...,6,8215.254237,424,0.766743,0.390528,0.399964,2,2,2,churn
764485,acb6894fa6835addaee68ff50a4dc461cae8cb22e83dc3...,6,8215.254237,355,0.719661,0.390528,0.399964,2,2,2,churn
582499,25b0da8f895c3b448bdd8a6c3018c70cf1d50fa81240c6...,6,8215.254237,629,0.913361,0.390528,0.399964,1,2,2,churn
285959,1763fc5c29be0c081c94d964510ef087410cbf328b83b6...,6,8215.254237,701,0.971076,0.390528,0.399964,1,2,2,churn
740971,7767ec071e99d67bfa0a1210d39129c91481a93aa958d3...,6,8215.254237,556,0.862468,0.390528,0.399964,1,2,2,churn
539762,c4d197a326185b9906671ea6b611918795be101702c036...,5,8215.254237,646,0.924797,0.341346,0.399964,1,2,2,churn
79318,4a4fd22ee1cfd03af19a7b8fb700e13b20926d3170c4f8...,6,8215.254237,322,0.692741,0.390528,0.399964,2,2,2,churn
998897,2110cf53d85c6d45cd200082d1311a6dec6508535616d1...,6,8215.254237,268,0.634943,0.390528,0.399964,2,2,2,churn
639937,279864c61dd97adb68ac747bc6e62ba2aab38b162be1d1...,4,8215.254237,508,0.832713,0.280644,0.399964,1,2,2,churn
35990,580c54ef102d6b831678185ad95fa8710e3f2b805f2a91...,6,8215.254237,707,0.975495,0.390528,0.399964,1,2,2,churn


In [31]:
type(df)

pandas.core.frame.DataFrame

In [32]:
# Guardar resultado
df.to_parquet("lakehouse/churn.parquet", engine="pyarrow", index=False)